In [1]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [2]:
date = read_table("select * from sc_gold.dim_date")
age_group = read_table("select * from sc_gold.dim_agegroup")
qualification = read_table("select * from sc_gold.dim_qualification")

In [3]:
df = read_table(
    """
    select 
    a.year,
    a.age_group,
    a.qualification,
    a.total_graduate,
    b.total_outside_labor_force,
    c.unemp_rate
    from sc_bronze.dosm_graduates_age a
    left join sc_bronze.dosm_olf_age b
    on a.year = b.year and a.age_group = b.age_group and a.qualification = b.qualification
    left join sc_bronze.dosm_unemp_rate_age c
    on a.year = c.year and a.age_group = c.age_group and a.qualification = c.qualification  
    """
)

df.head()

,year,age_group,qualification,total_graduate,total_outside_labor_force,unemp_rate
0,2016,25 - 34,degree,975800.0,83700.0,4.2
1,2016,25 - 34,diploma,871500.0,95000.0,3.1
2,2016,35 - 44,degree,568400.0,25800.0,1.0
3,2016,35 - 44,diploma,408800.0,34400.0,1.6
4,2016,≤ 24,degree,150300.0,31600.0,26.9


In [4]:
df["date"] = pd.to_datetime(df["year"], format="%Y")
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    age_group[["age_group", "age_group_id"]],
    on="age_group",
    how="left"
)

df = df.merge(
    qualification[["qualification", "qualification_id"]],
    on="qualification",
    how="left"
)

df_final = df.drop(columns=["year", "date", "age_group", "qualification"])
id_cols = ["date_id", "age_group_id", "qualification_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [5]:
df_final["daq_id"] = ["DAQ" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["daq_id"] + [c for c in df_final.columns if c != "daq_id"]]
df_final

,daq_id,date_id,age_group_id,qualification_id,total_graduate,total_outside_labor_force,unemp_rate
0,DAQ0001,DT001,AG002,Q001,975800.0,83700.0,4.2
1,DAQ0002,DT001,AG002,Q002,871500.0,95000.0,3.1
2,DAQ0003,DT001,AG003,Q001,568400.0,25800.0,1.0
3,DAQ0004,DT001,AG003,Q002,408800.0,34400.0,1.6
4,DAQ0005,DT001,AG001,Q001,150300.0,31600.0,26.9
...,...,...,...,...,...,...,...
67,DAQ0068,DT017,AG001,Q002,432200.0,164300.0,14.1
68,DAQ0069,DT021,AG001,Q002,447800.0,183600.0,13.6
69,DAQ0070,DT025,AG001,Q002,448600.0,186700.0,15.1
70,DAQ0071,DT029,AG001,Q002,471000.0,189900.0,15.9


In [6]:
write_table(df_final, "sc_gold", "fact_date_age_qualification")

Table sc_gold.fact_date_age_qualification written successfully.
